# DFU V4 Self-Contained Durable Missing-Trial Runner
No git clone, no bulk pip upgrade, no payload patching. Uses the pinned training source and skips already recovered evidence.


In [ ]:
# DFU V4 SELF-CONTAINED MISSING-TRIAL RUNNER
# No git clone. No bulk pip upgrade. No payload patching.

import os, sys, json, time, shutil, tarfile, urllib.request, importlib, subprocess, pickle
from pathlib import Path

PINNED_CODE_COMMIT = "4472d66a4f918d42ce9cfdfb57ed6dd95bdb0f11"
ALGORITHM_SOURCE_COMMIT = "349143b4d8b16f885adce3559542f6c202a2bca1"
RUN_ID = "RELIABLE_DFU_CV_V3_MISSING38"
DRIVE_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard")
BACKUP_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard-Backup")
RUN_ROOT = DRIVE_ROOT / "runs" / RUN_ID
REPO_ROOT = Path("/content/DFU-ImageGuard-pinned")
ARCHIVE = Path("/content/DFU-ImageGuard-pinned.tar.gz")
EXTRACT_ROOT = Path("/content/DFU-ImageGuard-pinned-extract")

print("=" * 72)
print("DFU V4 SELF-CONTAINED DURABLE RUNNER")
print("No git clone | No bulk pip | No payload patching")
print("=" * 72)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as e:
    raise RuntimeError(f"Google Drive mount failed: {type(e).__name__}: {e}")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
sentinel = RUN_ROOT / "V4_DRIVE_SENTINEL.txt"
token = f"{time.time_ns()}"
sentinel.write_text(token, encoding="utf-8")
if sentinel.read_text(encoding="utf-8") != token:
    raise RuntimeError("Drive read/write verification failed.")
print("Drive verification: PASS")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required. Colab: Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))

required = [
    ("timm", "timm"), ("kagglehub", "kagglehub"), ("imagehash", "ImageHash"),
    ("sklearn", "scikit-learn"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
    ("pandas", "pandas"), ("PIL", "Pillow"), ("tabulate", "tabulate"),
]
for module_name, package_name in required:
    try:
        importlib.import_module(module_name)
        print(f"dependency {module_name}: PASS")
        continue
    except Exception:
        print(f"dependency {module_name}: missing; installing only {package_name}")
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-input", "--disable-pip-version-check", package_name],
        text=True, capture_output=True
    )
    if p.returncode != 0:
        print(p.stdout[-2000:])
        print(p.stderr[-4000:])
        raise RuntimeError(f"Dependency install failed: {package_name}")
    importlib.invalidate_caches()
    importlib.import_module(module_name)
print("Dependency verification: PASS")

shutil.rmtree(REPO_ROOT, ignore_errors=True)
shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
ARCHIVE.unlink(missing_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
archive_url = "https://codeload.github.com/AzizulHakim00/DFU-ImageGuard/tar.gz/" + PINNED_CODE_COMMIT
print("Downloading pinned source archive...")
last_download_error = None
for attempt in range(1, 4):
    try:
        ARCHIVE.unlink(missing_ok=True)
        with urllib.request.urlopen(archive_url, timeout=120) as response, ARCHIVE.open("wb") as out:
            shutil.copyfileobj(response, out, length=8 * 1024 * 1024)
        if ARCHIVE.stat().st_size < 1024:
            raise RuntimeError(f"Downloaded archive is unexpectedly small: {ARCHIVE.stat().st_size} bytes")
        last_download_error = None
        break
    except Exception as e:
        last_download_error = e
        print(f"source download attempt {attempt}/3 failed: {type(e).__name__}: {e}")
        time.sleep(2 * attempt)
if last_download_error is not None:
    raise RuntimeError(f"Pinned source archive download failed after 3 attempts: {type(last_download_error).__name__}: {last_download_error}")

with tarfile.open(ARCHIVE, "r:gz") as tf:
    root = EXTRACT_ROOT.resolve()
    for member in tf.getmembers():
        target = (EXTRACT_ROOT / member.name).resolve()
        if target != root and root not in target.parents:
            raise RuntimeError(f"Unsafe archive member: {member.name}")
    tf.extractall(EXTRACT_ROOT)
folders = [p for p in EXTRACT_ROOT.iterdir() if p.is_dir()]
if len(folders) != 1:
    raise RuntimeError(f"Unexpected source archive layout: {folders}")
shutil.move(str(folders[0]), str(REPO_ROOT))
ARCHIVE.unlink(missing_ok=True)
shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
required_source = [
    REPO_ROOT / "src" / "reliable_runner_v2.py",
    REPO_ROOT / "src" / "reliable_storage_rescue.py",
    REPO_ROOT / "src" / "reliable_analysis.py",
]
missing_source = [str(p) for p in required_source if not p.is_file()]
if missing_source:
    raise RuntimeError(f"Pinned source incomplete: {missing_source}")
sys.path.insert(0, str(REPO_ROOT))
print("Pinned source preparation: PASS")

import pandas as pd
from src import reliable_runner_v2 as rr
from src.reliable_storage_rescue import storage_bounded_torch_save, metadata_only_active_backup
from src.reliable_analysis import build_reports

rr.atomic_torch = storage_bounded_torch_save
rr.backup_active_trial = metadata_only_active_backup

settings = rr.ReliableSettingsV2(
    run_id=RUN_ID,
    drive_root=str(DRIVE_ROOT),
    backup_root=str(BACKUP_ROOT),
    source_commit=ALGORITHM_SOURCE_COMMIT,
)
cfg = rr.build_config(settings)
EXPECTED = [(model, int(seed), int(fold)) for fold in settings.folds for seed in settings.seeds for model in settings.models]
if len(EXPECTED) != 45:
    raise RuntimeError(f"Protocol identity count changed unexpectedly: {len(EXPECTED)}")

def trial_path(model, seed, fold_zero):
    return RUN_ROOT / "trials" / model / f"seed_{seed}" / f"fold_{fold_zero + 1}"

def load_evidence(model, seed, fold_zero):
    t = trial_path(model, seed, fold_zero)
    cp, pp = t / "COMPLETE.json", t / "test_predictions.csv"
    if not (cp.is_file() and pp.is_file()):
        return None
    try:
        metrics = json.loads(cp.read_text(encoding="utf-8"))
        pred = pd.read_csv(pp)
        if pred.empty:
            return None
        ef = fold_zero + 1
        if str(metrics.get("model_key")) != model or int(metrics.get("seed")) != seed or int(metrics.get("outer_fold")) != ef:
            return None
        need = {"image_id","group_id","label","model_key","seed","outer_fold","prob_calibrated","pred"}
        if not need.issubset(pred.columns):
            return None
        ids = pred[["model_key","seed","outer_fold"]].drop_duplicates()
        if len(ids) != 1:
            return None
        row = ids.iloc[0]
        if str(row["model_key"]) != model or int(row["seed"]) != seed or int(row["outer_fold"]) != ef:
            return None
        return metrics, pred
    except Exception:
        return None

def atomic_csv_small(path, frame):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def atomic_json_small(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)

def rebuild_from_disk():
    metric_rows, pred_frames = [], []
    for model, seed, fold_zero in EXPECTED:
        found = load_evidence(model, seed, fold_zero)
        if found is None:
            continue
        metrics, pred = found
        metric_rows.append(metrics)
        pred_frames.append(pred)
    metrics_df = pd.DataFrame(metric_rows)
    preds_df = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    if not metrics_df.empty:
        metrics_df = metrics_df.sort_values(["outer_fold","seed","model_key"]).reset_index(drop=True)
    if not preds_df.empty:
        sort_cols = [c for c in ["outer_fold","seed","model_key","image_id"] if c in preds_df.columns]
        preds_df = preds_df.sort_values(sort_cols).reset_index(drop=True)
    atomic_csv_small(RUN_ROOT / "tables" / "fold_seed_metrics.csv", metrics_df)
    atomic_csv_small(RUN_ROOT / "tables" / "all_oof_predictions.csv", preds_df)
    return metrics_df, preds_df

metrics_df, preds_df = rebuild_from_disk()
print(f"Recovered durable evidence: {len(metrics_df)}/45 trials; prediction rows={len(preds_df)}")
if len(metrics_df) < 1:
    raise RuntimeError("No previously recovered trial evidence found; refusing blind restart.")
missing = [(m,s,f) for (m,s,f) in EXPECTED if load_evidence(m,s,f) is None]
print(f"Missing identities to train: {len(missing)}")
atomic_json_small(RUN_ROOT / "V4_PREFLIGHT.json", {
    "status":"PASS","run_id":RUN_ID,"pinned_code_commit":PINNED_CODE_COMMIT,
    "algorithm_source_commit":ALGORITHM_SOURCE_COMMIT,"existing_valid_trials":len(metrics_df),
    "missing_trials":len(missing),"prediction_rows":len(preds_df),"created_at_ns":time.time_ns(),
})
print("V4 preflight: PASS")

dirs = {"root": RUN_ROOT, **{name: RUN_ROOT / name for name in (
    "tables","figures","models","xai","predictions","logs","configs","manifests","cache"
)}}
for p in dirs.values():
    Path(p).mkdir(parents=True, exist_ok=True)

dataset_root = rr.download_dataset(cfg, dirs)
manifest = rr.build_manifest(dataset_root, cfg, dirs)
cleaned = rr.assign_duplicate_groups(manifest, cfg, dirs)
data = rr.make_outer_folds(cleaned, cfg, dirs)
print("Dataset/fold preparation: PASS")

try:
    for fold in settings.folds:
        outer_train = data[data.outer_fold != fold].copy()
        test_df = data[data.outer_fold == fold].copy().reset_index(drop=True)
        inner = rr.make_inner_partition(outer_train, cfg, fold)
        train_df = inner[inner.inner_role == "train"].copy()
        selection_df = inner[inner.inner_role == "selection"].copy()
        calibration_df = inner[inner.inner_role == "calibration"].copy()
        for seed in settings.seeds:
            for model_key in settings.models:
                if load_evidence(model_key, int(seed), int(fold)) is not None:
                    print(f"SKIP durable evidence: {model_key} seed={seed} fold={fold+1}")
                    continue
                t = trial_path(model_key, int(seed), int(fold))
                t.mkdir(parents=True, exist_ok=True)
                print("\n" + "-" * 72)
                print(f"TRAIN MISSING: {model_key} seed={seed} fold={fold+1}")
                print("-" * 72)
                rr.train_trial(
                    train_df=train_df, selection_df=selection_df, calibration_df=calibration_df,
                    test_df=test_df, model_key=model_key, seed=int(seed), fold=int(fold),
                    cfg=cfg, settings=settings, trial=t, run=RUN_ROOT,
                )
                if load_evidence(model_key, int(seed), int(fold)) is None:
                    raise RuntimeError(f"Durable evidence validation failed after trial: {model_key} seed={seed} fold={fold+1}")
                metrics_df, preds_df = rebuild_from_disk()
                atomic_json_small(RUN_ROOT / "V4_PROGRESS.json", {
                    "status":"RUNNING","completed_unique_trials":len(metrics_df),"expected_trials":45,
                    "prediction_rows":len(preds_df),"last_completed":{"model_key":model_key,"seed":int(seed),"outer_fold":int(fold)+1},
                    "updated_at_ns":time.time_ns(),
                })
                print(f"DURABLE SAVE VERIFIED: {len(metrics_df)}/45")
except Exception as e:
    metrics_df, preds_df = rebuild_from_disk()
    atomic_json_small(RUN_ROOT / "V4_STOP_REPORT.json", {
        "status":"INCOMPLETE_RESUMABLE","completed_unique_trials":len(metrics_df),"expected_trials":45,
        "prediction_rows":len(preds_df),"error":f"{type(e).__name__}: {e}","updated_at_ns":time.time_ns(),
    })
    print("RUN STOPPED:", f"{type(e).__name__}: {e}")
    print(f"Saved durable evidence remains: {len(metrics_df)}/45")
    raise

metrics_df, preds_df = rebuild_from_disk()
if len(metrics_df) != 45:
    raise RuntimeError(f"Training loop ended but only {len(metrics_df)}/45 durable trials exist.")
identity_count = metrics_df[["model_key","seed","outer_fold"]].drop_duplicates().shape[0]
if identity_count != 45:
    raise RuntimeError(f"Duplicate/missing identity conflict: unique={identity_count}, rows={len(metrics_df)}")

summary = metrics_df.groupby("model_key").agg({
    "balanced_accuracy":["mean","std"],"sensitivity":["mean","std"],"specificity":["mean","std"],
    "roc_auc":["mean","std"],"pr_auc":["mean","std"],"brier":["mean","std"],"ece":["mean","std"],
})
summary.to_csv(RUN_ROOT / "tables" / "model_summary.csv")
try:
    reports = build_reports(RUN_ROOT)
    report_status = "PASS"
except Exception as report_exc:
    reports = {"error": f"{type(report_exc).__name__}: {report_exc}"}
    report_status = "FAILED_REGENERATABLE"
    atomic_json_small(RUN_ROOT / "V4_REPORT_WARNING.json", {
        "status": report_status,
        "error": reports["error"],
        "note": "All 45 durable trial metrics/predictions are complete. Re-running this notebook will skip training and retry report generation.",
        "updated_at_ns": time.time_ns(),
    })
    print("REPORT GENERATION WARNING:", reports["error"])
with (RUN_ROOT / "reliable_dfu_reproducibility_v4.pkl").open("wb") as handle:
    pickle.dump({
        "run_id":RUN_ID,"pinned_code_commit":PINNED_CODE_COMMIT,"algorithm_source_commit":ALGORITHM_SOURCE_COMMIT,
        "settings":settings.__dict__,"metrics":metrics_df.to_dict("records"),"predictions":preds_df.to_dict("records"),
        "reports":reports,"note":"Recovered pre-existing trials use verified COMPLETE.json + test_predictions.csv evidence; no model weights are fabricated for them.",
    }, handle, pickle.HIGHEST_PROTOCOL)
final = {
    "status":"PASS","run_id":RUN_ID,"completed_unique_trials":45,"expected_trials":45,
    "prediction_rows":len(preds_df),"pinned_code_commit":PINNED_CODE_COMMIT,
    "algorithm_source_commit":ALGORITHM_SOURCE_COMMIT,"reports":reports,"report_status":report_status,"updated_at_ns":time.time_ns(),
}
atomic_json_small(RUN_ROOT / "V4_FINAL_VERIFICATION.json", final)
print(json.dumps(final, indent=2, default=str))

export_root = Path("/content/DFU_V4_EVIDENCE_EXPORT")
shutil.rmtree(export_root, ignore_errors=True)
export_root.mkdir(parents=True, exist_ok=True)
for rel in [
    "tables/fold_seed_metrics.csv","tables/all_oof_predictions.csv","tables/model_summary.csv",
    "tables/selective_prediction.csv","tables/error_audit.csv","tables/paired_bootstrap.json",
    "V4_PREFLIGHT.json","V4_PROGRESS.json","V4_FINAL_VERIFICATION.json","reliable_dfu_reproducibility_v4.pkl",
]:
    src = RUN_ROOT / rel
    if src.is_file():
        dst = export_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
zip_path = shutil.make_archive("/content/DFU_V4_EVIDENCE_EXPORT", "zip", export_root)
print("Evidence ZIP:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass
